# Dò tham số sinh văn bản cho BARTpho

Notebook này **không huấn luyện gì cả**. Nó nạp checkpoint BARTpho `train_20k` đã có,
rồi sinh lại bản tóm tắt trên tập `tune` với vài bộ tham số sinh khác nhau và chấm
điểm từng bộ. Cùng khuôn với `notebooks/sweep/` nhưng cho mô hình khác.

## Vì sao phải làm riêng cho BARTpho

Khảo sát tham số sinh ở tuần 5 chạy trên checkpoint **ViT5**, và kết luận "không cấu
hình nào thắng được mặc định" vì thế chỉ được chứng minh cho ViT5. Nhưng BARTpho mới
là mô hình tốt nhất của đề tài (35,23 so với 33,40 ROUGE-1 trên `val`), nên kết luận
ấy cần được đo lại trên chính nó chứ không suy diễn.

## Vì sao lưới tham số khác lưới của ViT5

Lưới của ViT5 chỉ dò **về phía sinh dài hơn**, vì ViT5 sinh 30-31 âm tiết trong khi
sapo thật dài 35 — tức đang hụt. BARTpho sinh **34 âm tiết**, gần như trúng đích, nên
câu hỏi đổi hẳn: không còn là "dài thêm có tốt hơn không" mà là **"mặc định đã nằm ở
điểm tối ưu chưa"**. Câu đó chỉ trả lời được bằng lưới **đối xứng hai phía**, nên ở
đây `length_penalty` chạy từ 0,6 tới 2,0 quanh mốc 1,0.

Ba mức 1,0 / 1,5 / 2,0 giữ nguyên của ViT5 để phía "dài hơn" còn so được giữa hai mô
hình; ba mức 0,6 / 0,8 / 1,2 là phần thêm mới.

`min_length` chỉ giữ **một** cấu hình thay vì ba. Trên ViT5, `min_length=20` hoá ra
không ràng buộc gì: bản tóm tắt mặc định đã dài trung bình 36 token và chỉ 3/500 bản
ngắn hơn 20 token, nên `min20` trùng 495/500 bản với mặc định. BARTpho còn sinh dài
hơn ViT5, nên ngưỡng ấy càng không thể chạm tới phân phối thật. Giữ lại một cấu hình
là đủ để xác nhận điều đó chứ không tiêu GPU cho ba.

## Trước khi chạy: Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |

Checkpoint đến từ output của notebook `dl-summarisevn-vit5` (khai trong
`kernel-metadata.json`, mục `kernel_sources`). Kaggle gắn **version mới nhất** của
notebook đó, và version mới nhất hiện là lần chạy BARTpho — nên ô tìm checkpoint ở
dưới trỏ đúng `vinai_bartpho-syllable_train_20k/final`.

**Cảnh báo về thứ tự:** nếu có ai đẩy một version mới lên kernel `dl-summarisevn-vit5`
trước khi notebook này chạy, `kernel_sources` sẽ gắn version mới đó và checkpoint
BARTpho không còn tự lấy được nữa — đúng như chuyện đã xảy ra với checkpoint ViT5.
Chạy notebook này **trước** mọi lần huấn luyện mới.


In [ ]:
# ==== CHI SUA O NAY ===================================================
EVAL_SPLIT = "tune"                      # KHONG doi thanh val hay test
NAME = "bartpho-syllable-train_20k"      # ten he thong trong bang ket qua
CKPT_GLOB = "/kaggle/input/**/vinai_bartpho-syllable_train_20k/final"

# Luoi DOI XUNG hai phia quanh mac dinh 1,0. BARTpho sinh 34 am tiet, sapo that dai
# 35 — gan nhu trung dich — nen cau hoi la "mac dinh da toi uu chua", khong phai
# "dai them co tot hon khong" nhu o ViT5.
#   length_penalty < 1 : beam search uu tien chuoi NGAN hon
#   length_penalty > 1 : uu tien chuoi DAI hon
#   min_length         : chan cung do dai toi thieu (token)
GRID = [
    {"length_penalty": 0.6, "min_length": 0},
    {"length_penalty": 0.8, "min_length": 0},
    {"length_penalty": 1.0, "min_length": 0},    # mac dinh, de lam moc
    {"length_penalty": 1.2, "min_length": 0},
    {"length_penalty": 1.5, "min_length": 0},
    {"length_penalty": 2.0, "min_length": 0},
    # Mot cau hinh min_length de XAC NHAN no khong rang buoc gi, nhu da thay o ViT5
    # (495/500 ban trung khit mac dinh). Khong dang tieu GPU cho ba cau hinh.
    {"length_penalty": 1.0, "min_length": 20},
]
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"
OUT = "/kaggle/working/sweep"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print(f"{len(GRID)} cau hinh tren tap {EVAL_SPLIT}")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Khong ra duoc Internet. Settings > Internet > On.") from e
print("Moi truong: OK")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1
print("cwd:", os.getcwd())

In [ ]:
# Tim checkpoint trong /kaggle/input. Bao loi RO RANG neu khong thay: nguyen nhan gan
# nhu luon la quen gan output cua notebook huan luyen lam input.
import glob

found = sorted(glob.glob(CKPT_GLOB, recursive=True))
if not found:
    co_gi = sorted(glob.glob("/kaggle/input/*/*"))[:20]
    raise RuntimeError(
        f"Khong thay checkpoint khop {CKPT_GLOB}.\n"
        "Vao Add-ons > Add data > Your Work, them output cua notebook "
        "dl-summarisevn-vit5 (ban chay BARTpho train_20k).\n"
        f"Hien /kaggle/input co: {co_gi}"
    )
CKPT = found[0]
print("Checkpoint:", CKPT)
!ls -la {CKPT} | head -8

## Chạy lưới tham số

Mỗi cấu hình là một lần gọi `vit5.py --no-train`: nạp checkpoint, sinh 500 bản tóm tắt,
chấm điểm, ghi ra `results/`. Tên file mang theo `lp` và `min` nên các cấu hình không
đè lên nhau; cấu hình mặc định (`lp=1.0, min=0`) không có hậu tố nào.

Không dùng `--eval-limit`: chấm thiếu bài thì các cấu hình không so cặp được với nhau.

In [ ]:
import time

t0 = time.time()
for i, g in enumerate(GRID, 1):
    cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --no-train "
           f"--model {CKPT} --name {NAME} --eval-split {EVAL_SPLIT} "
           f"--length-penalty {g['length_penalty']} --min-length {g['min_length']} "
           f"--out {OUT}")
    print(f"\n===== [{i}/{len(GRID)}] {g} =====")
    print(cmd)
    !{cmd}
    check(_exit_code, f"cau hinh {i} {g}")
print(f"\nXong {len(GRID)} cau hinh trong {(time.time() - t0) / 60:.1f} phut.")

In [ ]:
# Bang so sanh cac cau hinh. `table()` la dung ham cham diem cua du an, khong tu tinh lai.
import json, pathlib, sys
sys.path.insert(0, "src")
from eval.report import table

rows = []
for p in sorted(pathlib.Path("results/tables").glob(f"{NAME}_{EVAL_SPLIT}_in1024*.json")):
    if p.name.endswith("_run.json"):
        continue
    r = json.loads(p.read_text(encoding="utf-8"))[0]
    # Nhan hau to cua ten file lam ten hang: do la cau hinh sinh ra no.
    hau_to = p.stem.replace(f"{NAME}_{EVAL_SPLIT}_in1024", "").strip("_") or "mac dinh"
    rows.append({**r, "name": hau_to})
print(table(rows))
print("\nSapo that dai trung binh 35 am tiet — cot 'Do dai' cang gan 35 cang tot,")
print("nhung tieu chi chon van la rouge1/rouge2, do dai chi de giai thich.")

In [ ]:
# Gom ket qua (khong gom trong so) thanh mot zip de tai ve tu tab Output.
import zipfile

picked = sorted(p for p in pathlib.Path("results").rglob("*.json") if p.name.startswith(f"{NAME}_{EVAL_SPLIT}_"))
if not picked:
    raise RuntimeError("Khong thay file ket qua nao cua khao sat tham so.")
zpath = f"/kaggle/working/ket_qua_sweep_{NAME}_{EVAL_SPLIT}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():86s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)

## Sau khi chạy

1. Tải `ket_qua_sweep_*.zip` ở tab **Output**, giải nén tại thư mục gốc repo.
2. So các cấu hình bằng bootstrap ghép cặp (`eval.report.compare`) — chênh lệch nhỏ
   giữa hai cấu hình rất dễ là nhiễu, và cả sáu đều chấm trên đúng 500 bài của `tune`
   nên ghép cặp hợp lệ.
3. Cấu hình thắng mới được đem sang `val`/`test`. **Đừng** chọn cấu hình dựa trên `val`.